# LLM Prompting Methods for Healthcare Text & Structured Data

This tutorial compares four prompting strategies — **Zero-shot, Few-shot, Chain-of-Thought (CoT), and Tree-of-Thoughts (ToT)** — on the *same* classification goal across two different data shapes:

1. **MTSamples** (free-text clinical transcriptions) → predict `medical_specialty`
2. **Synthea** (structured synthetic EHR: conditions + medications) → predict opioid-misuse risk (Low/High), using a later recorded "Drug overdose" condition as a proxy ground-truth label

**Ground truth**
- MTSamples: the real `medical_specialty` column already in the dataset (not invented).
- Synthea: a *proxy* label — patients with a later recorded "Drug overdose" condition are labeled High risk. This is bounded by what the Synthea simulator generated, not a validated clinical risk score. Stated explicitly here for transparency.

**Data note:** MTSamples and Synthea CSVs are both public/synthetic and contain no real patient data, consistent with assignment requirements.

## 1. Setup

In [4]:
!pip install openai pandas scikit-learn --quiet


In [5]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = ""
gv_client = OpenAI()  # reads the API key from the OPENAI_API_KEY environment variable
gv_model = "gpt-4o-mini"


## 2. Load MTSamples eval set + ground truth

`mtsamples_eval_set.csv` is a stratified sample: 7 records each from 7 specialties
(Surgery, Cardiovascular/Pulmonary, Orthopedic, Radiology, Neurology, Urology,
Gastroenterology) — 49 records total.

`mtsamples_fewshot_pool.csv` holds a separate, non-overlapping set of 14 records
(2 per specialty) used only to build few-shot examples.

In [6]:
gv_mt_eval = pd.read_csv("/content/data/mtsamples_eval_set.csv")
gv_mt_fewshot_pool = pd.read_csv("/content/data/mtsamples_fewshot_pool.csv")

gv_specialties = sorted(gv_mt_eval["medical_specialty"].unique().tolist())

print("Ground truth — MTSamples specialty counts (eval set):")
print(gv_mt_eval["medical_specialty"].value_counts())
print("\nTotal eval records:", len(gv_mt_eval))
print("Specialty label set:", gv_specialties)


Ground truth — MTSamples specialty counts (eval set):
medical_specialty
Surgery                       7
Cardiovascular / Pulmonary    7
Orthopedic                    7
Radiology                     7
Neurology                     7
Urology                       7
Gastroenterology              7
Name: count, dtype: int64

Total eval records: 49
Specialty label set: ['Cardiovascular / Pulmonary', 'Gastroenterology', 'Neurology', 'Orthopedic', 'Radiology', 'Surgery', 'Urology']


## 3. Load Synthea eval set + ground truth

`synthea_eval_ids.json` holds patient IDs for 25 overdose-positive and 25
negative patients (eval set), plus a separate 4+4 few-shot pool, and each
positive patient's overdose date (used to truncate their history so the
model never sees the overdose itself — avoiding label leakage).

In [7]:
gv_patients = pd.read_csv("/content/data/patients.csv")
gv_conditions = pd.read_csv("/content/data/conditions.csv")
gv_medications = pd.read_csv("/content/data/medications.csv")

with open("/content/data/synthea_eval_ids.json") as f:
    gv_synthea_ids = json.load(f)

gv_overdose_dates = gv_synthea_ids["overdose_dates"]

def build_patient_text(lv_patient_id, lv_truncate_before=None):
    """Turn a patient's conditions and medications into plain text.
    If truncate_before is set, only include events before that date,
    so the model never sees the overdose itself."""
    lv_c = gv_conditions[gv_conditions["PATIENT"] == lv_patient_id][["START", "DESCRIPTION"]].copy()
    lv_m = gv_medications[gv_medications["PATIENT"] == lv_patient_id][["START", "DESCRIPTION"]].copy()
    lv_c["START"] = lv_c["START"].astype(str).str[:10]
    lv_m["START"] = lv_m["START"].astype(str).str[:10]

    if lv_truncate_before:
        lv_cutoff = lv_truncate_before[:10]
        lv_c = lv_c[lv_c["START"] < lv_cutoff]
        lv_m = lv_m[lv_m["START"] < lv_cutoff]

    lv_c = lv_c.sort_values("START").drop_duplicates("DESCRIPTION")
    lv_m = lv_m.sort_values("START").drop_duplicates("DESCRIPTION")

    lv_lines = ["CONDITIONS (chronological):"]
    for _, lv_r in lv_c.iterrows():
        lv_lines.append(f"  {lv_r['START']}: {lv_r['DESCRIPTION']}")
    lv_lines.append("MEDICATIONS (chronological):")
    for _, lv_r in lv_m.iterrows():
        lv_lines.append(f"  {lv_r['START']}: {lv_r['DESCRIPTION']}")
    return "\n".join(lv_lines)

# Build the eval set: one row per patient with their text and label
gv_synthea_eval = []
for gv_pid in gv_synthea_ids["eval_positive"]:
    gv_txt = build_patient_text(gv_pid, gv_overdose_dates.get(gv_pid))
    gv_synthea_eval.append({"patient_id": gv_pid, "text": gv_txt, "ground_truth": "High"})
for gv_pid in gv_synthea_ids["eval_negative"]:
    gv_txt = build_patient_text(gv_pid, None)
    gv_synthea_eval.append({"patient_id": gv_pid, "text": gv_txt, "ground_truth": "Low"})

gv_synthea_eval_df = pd.DataFrame(gv_synthea_eval)

print("Ground truth — Synthea risk label counts (eval set):")
print(gv_synthea_eval_df["ground_truth"].value_counts())
print("\nTotal eval patients:", len(gv_synthea_eval_df))


Ground truth — Synthea risk label counts (eval set):
ground_truth
High    25
Low     25
Name: count, dtype: int64

Total eval patients: 50


## 4. Build few-shot example blocks

Few-shot examples are drawn only from the separate pools (never the eval set)
to avoid the model simply memorizing eval answers.

In [8]:
# Few-shot examples for MTSamples
def build_mtsamples_fewshot_block(lv_n_per_class=1):
    lv_lines = []
    for lv_spec in gv_specialties:
        lv_rows = gv_mt_fewshot_pool[gv_mt_fewshot_pool["medical_specialty"] == lv_spec].head(lv_n_per_class)
        for _, lv_r in lv_rows.iterrows():
            lv_snippet = str(lv_r["transcription"])[:600]
            lv_lines.append(f'Report: """{lv_snippet}"""\nSpecialty: {lv_spec}\n')
    return "\n".join(lv_lines)

gv_mt_fewshot_block = build_mtsamples_fewshot_block(1)

# Few-shot examples for Synthea
def build_synthea_fewshot_block():
    lv_lines = []
    for lv_pid in gv_synthea_ids["fewshot_positive"]:
        lv_txt = build_patient_text(lv_pid, gv_overdose_dates.get(lv_pid))
        lv_lines.append(f'Patient history:\n{lv_txt}\nRisk: High\n')
    for lv_pid in gv_synthea_ids["fewshot_negative"]:
        lv_txt = build_patient_text(lv_pid, None)
        lv_lines.append(f'Patient history:\n{lv_txt}\nRisk: Low\n')
    return "\n".join(lv_lines)

gv_synthea_fewshot_block = build_synthea_fewshot_block()
print(gv_mt_fewshot_block[:800])


Report: """PREOPERATIVE DIAGNOSES:,1.  Ischemic cardiomyopathy.,2.  Status post redo coronary artery bypass.,3.  Status post insertion of intraaortic balloon.,POSTOPERATIVE DIAGNOSES:,1.  Ischemic cardiomyopathy.,2.  Status post redo coronary artery bypass.,3.  Status post insertion of intraaortic balloon.,4.  Postoperative coagulopathy.,OPERATIVE PROCEDURE:,1.  Orthostatic cardiac allograft transplantation utilizing total cardiopulmonary bypass.,2.  Open sternotomy covered with Ioban.,3.  Insertion of Mahurkar catheter for hemofiltration via the left common femoral vein.,ANESTHESIA: , General endotrache"""
Specialty: Cardiovascular / Pulmonary

Report: """DIAGNOSIS ON ADMISSION: , Gastrointestinal bleed.,DIAGNOSES ON DISCHARGE,1. Gastrointestinal bleed, source undetermined, but possibly d


## 5. Prompt templates

Every method / dataset combination returns the **same fixed JSON schema**:
`{"label": "...", "reason": "..."}` — this is what makes the four methods
directly comparable.

In [9]:
def mtsamples_prompt(lv_method, lv_transcription):
    lv_schema_note = 'Respond ONLY with JSON: {"label": "<one specialty>", "reason": "<one sentence>"}'
    lv_specialty_list = ", ".join(gv_specialties)

    if lv_method == "zero_shot":
        return f"""Classify the medical specialty of this clinical report.
Choose exactly one from: {lv_specialty_list}.
{lv_schema_note}

Report:
\"\"\"{lv_transcription}\"\"\""""

    elif lv_method == "few_shot":
        return f"""Classify the medical specialty of a clinical report.
Choose exactly one from: {lv_specialty_list}.
{lv_schema_note}

Examples:
{gv_mt_fewshot_block}

Now classify this report:
Report: \"\"\"{lv_transcription}\"\"\""""

    elif lv_method == "cot":
        return f"""Classify the medical specialty of this clinical report.
Choose exactly one from: {lv_specialty_list}.

First, silently reason step by step: (1) identify the key clinical findings,
(2) identify the procedure or diagnosis type, (3) match those to the closest
specialty. Then output ONLY the final JSON (do not show your reasoning steps
in the output).
{lv_schema_note}

Report:
\"\"\"{lv_transcription}\"\"\""""

    elif lv_method == "tot":
        return f"""Classify the medical specialty of this clinical report.
Choose exactly one from: {lv_specialty_list}.

Internally consider THREE plausible specialty interpretations based on the
report's findings, briefly weigh the evidence for each, then select the
best-supported one. Output ONLY the final JSON (do not show the three
interpretations in the output).
{lv_schema_note}

Report:
\"\"\"{lv_transcription}\"\"\""""

    else:
        raise ValueError(lv_method)


def synthea_prompt(lv_method, lv_patient_text):
    lv_schema_note = ('Respond ONLY with JSON: {"label": "Low" or "High", '
                    '"probability_high": <float 0.0-1.0, your confidence that risk is High>, '
                    '"reason": "<one sentence>"}')

    if lv_method == "zero_shot":
        return f"""Given this patient's condition and medication history, classify
their opioid-misuse risk as Low or High.
{lv_schema_note}

{lv_patient_text}"""

    elif lv_method == "few_shot":
        return f"""Given a patient's condition and medication history, classify
their opioid-misuse risk as Low or High.
{lv_schema_note}

Examples:
{gv_synthea_fewshot_block}

Now classify this patient:
{lv_patient_text}"""

    elif lv_method == "cot":
        return f"""Given this patient's condition and medication history, classify
their opioid-misuse risk as Low or High.

First, silently reason step by step: (1) list the conditions in chronological
order, (2) list the medications in chronological order, (3) note any pattern
of escalation, switching, or discontinuation in medication type/potency over
time, (4) classify risk based on that pattern. Then output ONLY the final
JSON (do not show your reasoning steps in the output).
{lv_schema_note}

{lv_patient_text}"""

    elif lv_method == "tot":
        return f"""Given this patient's condition and medication history, classify
their opioid-misuse risk as Low or High.

Internally consider THREE possible explanations for this timeline:
(a) normal short-term pain management, (b) tolerance-driven escalation,
(c) unrelated/incidental events. Briefly weigh each against the timeline,
then classify risk based on the best-supported explanation. Output ONLY the
final JSON (do not show the three explanations in the output).
{lv_schema_note}

{lv_patient_text}"""

    else:
        raise ValueError(lv_method)


## 6. API call + parsing helper

Uses JSON mode where available and retries once on malformed output.

In [10]:
def call_llm(lv_prompt, lv_retries=2):
    """Call the model and return its answer as a dict.
    If it fails after retries, return an error label instead of crashing."""
    for lv_attempt in range(lv_retries + 1):
        try:
            lv_resp = gv_client.chat.completions.create(
                model=gv_model,
                messages=[{"role": "user", "content": lv_prompt}],
                temperature=0,
                response_format={"type": "json_object"},
            )
            lv_content = lv_resp.choices[0].message.content
            lv_parsed = json.loads(lv_content)
            lv_parsed["label"] = str(lv_parsed.get("label", "")).strip()
            lv_parsed["reason"] = str(lv_parsed.get("reason", "")).strip()
            if "probability_high" in lv_parsed:
                try:
                    lv_parsed["probability_high"] = float(lv_parsed["probability_high"])
                except (TypeError, ValueError):
                    lv_parsed["probability_high"] = None
            return lv_parsed
        except Exception as lv_e:
            if lv_attempt == lv_retries:
                print("Failed after retries:", lv_e)
                return {"label": "PARSE_ERROR", "reason": str(lv_e), "probability_high": None}
            time.sleep(1)


## 7. Run all methods on MTSamples (49 records x 4 methods = 196 calls)

In [11]:
gv_methods = ["zero_shot", "few_shot", "cot", "tot"]

gv_mt_results = []
for _, gv_row in gv_mt_eval.iterrows():
    for gv_method in gv_methods:
        gv_prompt = mtsamples_prompt(gv_method, gv_row["transcription"])
        gv_parsed = call_llm(gv_prompt)
        gv_mt_results.append({
            "record_id": gv_row["sample_name"],
            "method": gv_method,
            "predicted_label": gv_parsed["label"],
            "ground_truth": gv_row["medical_specialty"],
            "reason": gv_parsed["reason"],
        })

gv_mt_results_df = pd.DataFrame(gv_mt_results)
gv_mt_results_df.to_csv("mtsamples_results.csv", index=False)
gv_mt_results_df.head()


,record_id,method,predicted_label,ground_truth,reason
0,Myringotomy/Tube Insertion,zero_shot,Surgery,Surgery,The report describes a surgical procedure invo...
1,Myringotomy/Tube Insertion,few_shot,Surgery,Surgery,The report details a surgical procedure involv...
2,Myringotomy/Tube Insertion,cot,Surgery,Surgery,The report details a surgical procedure involv...
3,Myringotomy/Tube Insertion,tot,Surgery,Surgery,The report details a surgical procedure involv...
4,Triple Lumen Catheter Insertion,zero_shot,Surgery,Surgery,The report details surgical procedures involvi...


## 8. Run all methods on Synthea (50 patients x 4 methods = 200 calls)

In [12]:
gv_synthea_results = []
for _, gv_row in gv_synthea_eval_df.iterrows():
    for gv_method in gv_methods:
        gv_prompt = synthea_prompt(gv_method, gv_row["text"])
        gv_parsed = call_llm(gv_prompt)
        gv_synthea_results.append({
            "patient_id": gv_row["patient_id"],
            "method": gv_method,
            "predicted_label": gv_parsed["label"],
            "probability_high": gv_parsed.get("probability_high"),
            "ground_truth": gv_row["ground_truth"],
            "reason": gv_parsed["reason"],
        })

gv_synthea_results_df = pd.DataFrame(gv_synthea_results)
gv_synthea_results_df.to_csv("synthea_results.csv", index=False)
gv_synthea_results_df.head()


,patient_id,method,predicted_label,probability_high,ground_truth,reason
0,59cf17d9-6c13-4333-a1cb-cc5fdf63366d,zero_shot,High,0.85,High,The patient has a history of chronic pain and ...
1,59cf17d9-6c13-4333-a1cb-cc5fdf63366d,few_shot,High,0.85,High,The patient has a history of chronic pain and ...
2,59cf17d9-6c13-4333-a1cb-cc5fdf63366d,cot,High,0.85,High,The patient has a history of escalating opioid...
3,59cf17d9-6c13-4333-a1cb-cc5fdf63366d,tot,High,0.85,High,The patient's escalating use of opioids for ch...
4,2abf5d21-8d0f-4263-b720-81d9d25f7a70,zero_shot,Low,0.10,High,The patient's condition and medication history...


## 9. Summary Table 1 — MTSamples: predicted specialty counts by method

In [13]:
gv_mt_summary_counts = pd.crosstab(gv_mt_results_df["method"], gv_mt_results_df["predicted_label"])
print("Predicted label distribution by method:")
gv_mt_summary_counts


Predicted label distribution by method:


predicted_label,Cardiovascular,Cardiovascular / Pulmonary,Gastroenterology,Neurology,Orthopedic,Pulmonary,Radiology,Surgery,Urology
method,,,,,,,,,
cot,5,0,3,9,9,2,1,15,5
few_shot,0,7,3,9,7,0,3,14,6
tot,5,0,3,9,9,2,1,14,6
zero_shot,5,0,3,9,6,2,2,17,5


In [14]:
gv_mt_accuracy = (
    gv_mt_results_df.groupby("method")
    .apply(lambda lv_g: (lv_g["predicted_label"] == lv_g["ground_truth"]).mean())
    .rename("accuracy")
    .reset_index()
)
print("Accuracy by prompting method (MTSamples specialty classification):")
gv_mt_accuracy


Accuracy by prompting method (MTSamples specialty classification):


/tmp/ipykernel_825/3696268873.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda lv_g: (lv_g["predicted_label"] == lv_g["ground_truth"]).mean())


,method,accuracy
0,cot,0.551020
1,few_shot,0.673469
2,tot,0.530612
3,zero_shot,0.530612


## 10. Summary Table 2 — Synthea: predicted risk counts + confusion matrix by method

In [15]:
gv_synthea_summary_counts = pd.crosstab(gv_synthea_results_df["method"], gv_synthea_results_df["predicted_label"])
print("Predicted label distribution by method:")
gv_synthea_summary_counts


Predicted label distribution by method:


predicted_label,High,Low
method,,
cot,2,48
few_shot,29,21
tot,2,48
zero_shot,6,44


In [16]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

gv_rows = []
for gv_method in gv_methods:
    gv_sub = gv_synthea_results_df[gv_synthea_results_df["method"] == gv_method]
    gv_y_true = (gv_sub["ground_truth"] == "High").astype(int)
    gv_y_pred = (gv_sub["predicted_label"] == "High").astype(int)
    gv_tn, gv_fp, gv_fn, gv_tp = confusion_matrix(gv_y_true, gv_y_pred, labels=[0,1]).ravel()
    gv_rows.append({
        "method": gv_method,
        "TP (correctly caught High risk)": gv_tp,
        "FN (missed High risk)": gv_fn,
        "FP (false alarm)": gv_fp,
        "TN (correctly Low risk)": gv_tn,
        "precision": round(precision_score(gv_y_true, gv_y_pred, zero_division=0), 2),
        "recall": round(recall_score(gv_y_true, gv_y_pred, zero_division=0), 2),
        "accuracy": round(accuracy_score(gv_y_true, gv_y_pred), 2),
    })

gv_synthea_confusion_summary = pd.DataFrame(gv_rows)
print("Confusion matrix + precision/recall/accuracy by method (Synthea opioid-risk proxy):")
gv_synthea_confusion_summary


Confusion matrix + precision/recall/accuracy by method (Synthea opioid-risk proxy):


,method,TP (correctly caught High risk),FN (missed High risk),FP (false alarm),TN (correctly Low risk),precision,recall,accuracy
0,zero_shot,5,20,1,24,0.83,0.20,0.58
1,few_shot,21,4,8,17,0.72,0.84,0.76
2,cot,2,23,0,25,1.00,0.08,0.54
3,tot,2,23,0,25,1.00,0.08,0.54


## 11. Caveats and Limitations

- **MTSamples ground truth** is the dataset's own `medical_specialty` field —
  a real label, not invented for this project.
- **Synthea ground truth is a proxy**, not a validated clinical risk score:
  "High risk" = the simulated patient happened to have a later recorded
  "Drug overdose" condition in this specific Synthea run. Absence of that
  condition does not prove the patient was truly low-risk — Synthea's
  simulation may simply not have generated that outcome for them.
- **Positive cases were oversampled for evaluation.** Only 52 of 1,171
  patients (4.4%) in the underlying Synthea sample have a recorded overdose;
  the eval set uses 25 positives + 25 negatives so every method has a fair
  chance to be tested on both classes, rather than the natural imbalance
  hiding nearly all positives.
- All few-shot examples were drawn from a separate pool with zero overlap
  with the evaluation records, to avoid memorization inflating results.